# Lesson 06: 재질과 마찰 - 미끄러운 바닥 vs 거친 바닥

## 학습 목표
1. 서로 다른 마찰 계수(friction) 비교
2. 서로 다른 반발 계수(restitution) 비교
3. 경사면 만들기: 쿼터니언 회전(`SetRot`)
4. 같은 물체를 다른 조건에서 실험하는 법

## 새로 배우는 Chrono API
| 클래스/메서드 | 역할 |
|---|---|
| `ChQuaterniond()` | 회전을 표현하는 쿼터니언 |
| `q.SetFromAngleZ(rad)` | Z축 기준 회전 (경사면 만들기) |
| `body.SetRot(q)` | 물체에 회전 적용 |

> **쿼터니언**: 3D 회전을 표현하는 수학적 도구. 오일러 각도보다 짐벌락 없이 안정적.

In [ ]:
import math
import pychrono as chrono

sys = chrono.ChSystemNSC()
sys.SetGravitationalAcceleration(chrono.ChVector3d(0, -9.81, 0))
sys.SetCollisionSystemType(chrono.ChCollisionSystem.Type_BULLET)

## 실험 1: 마찰 계수 비교

같은 20도 경사면에서 다른 마찰 계수의 공을 굴림:
- 얼음 (friction=0.0): 저항 없이 미끄러짐
- 나무 (friction=0.5): 보통
- 고무 (friction=1.0): 강한 저항

In [ ]:
# 3가지 마찰 재질
mat_ice = chrono.ChContactMaterialNSC()
mat_ice.SetFriction(0.0)      # 얼음
mat_ice.SetRestitution(0.3)

mat_wood = chrono.ChContactMaterialNSC()
mat_wood.SetFriction(0.5)     # 나무
mat_wood.SetRestitution(0.3)

mat_rubber = chrono.ChContactMaterialNSC()
mat_rubber.SetFriction(1.0)   # 고무
mat_rubber.SetRestitution(0.3)

materials = [
    (mat_ice,    "얼음 (friction=0.0)"),
    (mat_wood,   "나무 (friction=0.5)"),
    (mat_rubber, "고무 (friction=1.0)"),
]

for _, name in materials:
    print(f"  {name}")

## 경사면 만들기: 쿼터니언 회전

- 상자를 Z축 기준으로 20도 회전하면 경사면이 됨
- `SetFromAngleZ(radian)` 사용 (도→라디안 변환 필요)

In [ ]:
ramp_angle = 20  # 도(degree)
ramp_rad = math.radians(ramp_angle)

balls = []

for i, (mat, name) in enumerate(materials):
    z_offset = (i - 1) * 4  # z = -4, 0, 4 (나란히 배치)

    # 경사면 (기울어진 상자)
    ramp = chrono.ChBody()
    ramp.SetPos(chrono.ChVector3d(-1, 2, z_offset))
    ramp.SetFixed(True)
    ramp.EnableCollision(True)

    # ★ 쿼터니언으로 Z축 기준 회전
    q = chrono.ChQuaterniond()
    q.SetFromAngleZ(ramp_rad)
    ramp.SetRot(q)

    ramp.AddCollisionShape(chrono.ChCollisionShapeBox(mat, 6, 0.3, 2))
    sys.AddBody(ramp)

    # 경사면 아래 평평한 바닥 (마찰 차이 관찰용)
    flat = chrono.ChBody()
    flat.SetPos(chrono.ChVector3d(5, 0, z_offset))
    flat.SetFixed(True)
    flat.EnableCollision(True)
    flat.AddCollisionShape(chrono.ChCollisionShapeBox(mat, 12, 0.3, 2))
    sys.AddBody(flat)

    # 경사면 위에 공 올려놓기
    ball_x = -1 - 2.0 * math.cos(ramp_rad)
    ball_y = 2 + 2.0 * math.sin(ramp_rad) + 0.4
    ball = chrono.ChBodyEasySphere(0.3, 2000, True, True, mat)
    ball.SetPos(chrono.ChVector3d(ball_x, ball_y, z_offset))
    sys.AddBody(ball)
    balls.append(ball)

    print(f"  경사면 {i+1} ({name}): z={z_offset}")

## 실험 2: 반발 계수 비교

같은 높이(6m)에서 떨어뜨려 튕김 정도 비교:
- 찰흙 (restitution=0.0): 에너지 완전 흡수
- 보통 (restitution=0.5): 절반 높이로 튕김
- 슈퍼볼 (restitution=0.95): 거의 원래 높이로 튕김

In [ ]:
mat_clay = chrono.ChContactMaterialNSC()
mat_clay.SetFriction(0.5)
mat_clay.SetRestitution(0.0)   # 찰흙

mat_normal = chrono.ChContactMaterialNSC()
mat_normal.SetFriction(0.5)
mat_normal.SetRestitution(0.5)  # 보통

mat_super = chrono.ChContactMaterialNSC()
mat_super.SetFriction(0.5)
mat_super.SetRestitution(0.95)  # 슈퍼볼

bounce_mats = [
    (mat_clay,   "찰흙 (restitution=0.0)"),
    (mat_normal, "보통 (restitution=0.5)"),
    (mat_super,  "슈퍼볼 (restitution=0.95)"),
]

bounce_balls = []
for i, (mat, name) in enumerate(bounce_mats):
    x_offset = 8 + i * 3

    # 바닥 패드
    pad = chrono.ChBody()
    pad.SetPos(chrono.ChVector3d(x_offset, -0.5, 0))
    pad.SetFixed(True)
    pad.EnableCollision(True)
    pad.AddCollisionShape(chrono.ChCollisionShapeBox(mat, 2, 1, 2))
    sys.AddBody(pad)

    # 공 (같은 높이에서 낙하)
    bb = chrono.ChBodyEasySphere(0.3, 2000, True, True, mat)
    bb.SetPos(chrono.ChVector3d(x_offset, 6, 0))
    sys.AddBody(bb)
    bounce_balls.append(bb)

    print(f"  {name}: x={x_offset}")

## 시뮬레이션 실행 + 결과 비교

In [ ]:
while sys.GetChTime() < 5.0:
    sys.DoStepDynamics(0.005)

print(f"시뮬레이션 {sys.GetChTime():.2f}초 완료")

print("\n=== 마찰 비교 (경사면) ===")
for i, (_, name) in enumerate(materials):
    b = balls[i]
    print(f"  {name}")
    print(f"    최종 위치: x={b.GetPos().x:.2f}, y={b.GetPos().y:.2f}")

print("\n=== 반발 비교 (낙하) ===")
for i, (_, name) in enumerate(bounce_mats):
    b = bounce_balls[i]
    print(f"  {name}")
    print(f"    최종 높이: y={b.GetPos().y:.3f} m")

## 정리: Phase 1 전체 API 총정리

| 레슨 | 핵심 API | 기능 |
|---|---|---|
| 01 | `ChSystemNSC`, `ChBody`, `DoStepDynamics` | 물리 세계 + 물체 + 시뮬레이션 |
| 02 | `ChContactMaterialNSC`, `ChCollisionShape*`, `EnableCollision` | 충돌 + 접촉 재질 |
| 03 | `ChVisualSystem*`, `ChVisualShape*`, `ChColor` | 3D 시각화 |
| 04 | `ChBodyEasySphere/Box/Cylinder` | 간편 물체 생성 (밀도→질량 자동) |
| 05 | `SetPosDt()` | 초기 속도 (물체 발사) |
| 06 | `ChQuaterniond`, `SetRot()` | 회전 (경사면), 재질 비교 실험 |

### 다음 Phase 2에서 배울 것
- **조인트(Joint)**: 물체끼리 연결 (경첩, 슬라이더)
- **스프링/댐퍼**: 탄성 연결
- **모터**: 자동 회전/이동
- → 로버 서스펜션, 드론 프로펠러의 기초